# Data loading and wrangling

Before any model, a dataset has to be understood and shaped. This notebook loads the California
Housing table, checks its structure and quality, derives a few features, and produces a clean
train/test split that the later notebooks reuse.

## Learning objectives

By the end of this notebook you will be able to:

- load a tabular dataset and inspect shape, types, and missing values;
- describe the target and its units;
- derive ratio features that carry more signal than raw counts;
- recognise outliers and decide whether to keep or trim them;
- make a reproducible train/test split with a fixed seed.

## Concept

**Wrangling** is the unglamorous work of turning a raw table into one a model can use. The first
steps never change: how many rows and columns, what type is each column, are there missing values,
and what do the ranges look like. A model fed nonsense is nonsense out, so these checks come
before any algorithm.

**Feature engineering** creates new columns from old ones. Raw counts such as `Population` are
often less useful than a ratio such as `rooms_per_person`, because a ratio is comparable across
neighbourhoods of different sizes. The module's `analysis.add_features` adds three such ratios.

**Outliers** are values far from the rest. They may be errors or they may be the most interesting
rows. Trimming them changes the question, so we inspect first and document the decision. Crucially,
any trimming or scaling must be learned on the training set only, or the test score leaks
information.

A **train/test split** holds back part of the data so the model is judged on rows it has never
seen. Fixing the random seed makes the split reproducible.

## Worked example

### Load and inspect

`load_california` reads `data/raw/california_housing.csv` (fetch it with
`python scripts/download_data.py --module 07`).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
import analysis
from ds_practice import load_california, set_seed

set_seed(42)
housing = load_california()
print("shape:", housing.shape)
display(housing.head())

In [ ]:
housing.info()
print("\nmissing values per column:")
print(housing.isna().sum())

### Target and ranges

`MedHouseVal` is the median house value for a census block group, measured in hundreds of thousands
of dollars. The summary below shows the target is capped at 5.0 (about $500k), which is a known
artefact of the original dataset.

In [ ]:
print(housing[analysis.TARGET].describe().round(3).to_string())
print("\nranges of the base features:")
display(housing[analysis.BASE_FEATURES].describe().loc[["min", "max"]].round(2))

### Feature engineering

Three ratios are added. `population_per_household` restates `AveOccup` in the same units, while
`bedrooms_per_room` and `rooms_per_person` describe crowding in a way raw counts cannot.

In [ ]:
housing = analysis.add_features(housing)
new_columns = [c for c in housing.columns if c not in analysis.BASE_FEATURES + [analysis.TARGET]]
print("added:", new_columns)
display(housing[new_columns].describe().round(3))

### Outliers

A few block groups report implausibly many rooms. We inspect rather than silently remove; if we do
trim, we record how many rows and why.

In [ ]:
extreme = housing.nlargest(5, "AveRooms")[["AveRooms", "AveOccup", "Population", "MedHouseVal"]]
display(extreme)

threshold = housing["AveRooms"].quantile(0.99)
trimmed = housing[housing["AveRooms"] <= threshold]
print(f"rows above the 99th percentile of AveRooms: {len(housing) - len(trimmed)}")

### A reproducible split

We keep all rows (outliers included) for teaching and split 80/20 with a fixed seed. The test set
is set aside now and not touched until evaluation.

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(housing, test_size=0.2, random_state=42)
print("train:", train.shape, "| test:", test.shape)
print("mean target train/test:",
      round(train[analysis.TARGET].mean(), 3),
      round(test[analysis.TARGET].mean(), 3))

## Exercises

1. **Range check.** Report how many block groups have `AveRooms` above 20 and what share of the
   data that is. Decide whether to drop them and justify the choice in two sentences.
2. **A new ratio.** Add `bedrooms_per_person = AveBedrms / AveOccup` to a copy of the frame and
   compare its correlation with the target to that of `AveBedrms`.
3. **Split sizes.** Re-run `train_test_split` with `test_size=0.3` and `random_state=7`. Report the
   train and test row counts and explain why fixing the seed matters.

## Limitations

The target is censored at 5.0, so any model under-predicts the most expensive areas. The features
are block-group aggregates, not individual homes, which is an ecological fallacy if read as
household facts. Coordinates encode geography but also stand in for unmeasured factors, so a model
can learn location-specific patterns that will not transfer to another region.